# Galaxy3 — Experimento 1: CNN propia entrenada desde cero

**Pregunta del bloque:** ¿qué estrategia resulta más adecuada cuando tenemos un conjunto
relativamente pequeño de imágenes astronómicas?

En este notebook construimos la **referencia**: una CNN propia (`GalaxyCNN`) entrenada desde
cero, y observamos sobre la misma arquitectura y el mismo protocolo el cambio asociado a
**BatchNorm** (run A → B) y a **Dropout** (run B → C).

El notebook es un *orquestador pedagógico*: todo el trabajo lo hace el paquete `galaxy3`
(datos, modelo, entrenamiento, métricas, figuras y MLflow). Aquí no hay training loops
propios; el notebook y la CLI (`python -m galaxy3.train`) llaman al mismo
`run_experiment(config)`.

**El experimento oficial está cerrado** (`docs/RESULTADOS.md`). Por defecto
(`TRAIN = True`) el notebook entrena A, B y C en tu propio experimento MLflow; con
`TRAIN = False` carga los runs oficiales, que solo existen en el MLflow del docente.

**El conjunto de test no participa en ninguna decisión.** Se evaluó una sola vez (Fase 13),
sobre la configuración elegida con validation. La sección 15 muestra esos resultados
leyéndolos de MLflow, sin volver a evaluar.

| Sección | Tema |
|---|---|
| 0 | Parámetros y ambiente |
| 1 | Pregunta experimental y protocolo |
| 2 | Datos: originales vs transformadas |
| 3 | Batches y shapes |
| 4 | Arquitectura y parámetros |
| 5 | BatchNorm: `train()` vs `eval()` |
| 6 | Dropout: `train()` vs `eval()` |
| 7 | Feature maps antes de entrenar |
| 8 | Qué mostró el piloto y por qué 40 épocas |
| 9 | Los tres runs |
| 10 | Curvas y comparación en validation |
| 11 | Loss vs accuracy: sobreconfianza y errores |
| 12 | MLflow |
| 13 | Feature maps después de entrenar |
| 14 | Cierre |
| 15 | Evaluación final en test (resultados registrados) |

## 0. Parámetros y ambiente

Esta es la **única celda que cambia** entre ejecuciones:

- `TRAIN = True` (por defecto): la sección 9 entrena A, B y C con `run_experiment`, igual
  que la CLI, en el experimento MLflow `EXPERIMENT_NAME` y en la carpeta `OUTPUT_ROOT`.
  `EPOCHS = 40` es el protocolo oficial; `NUM_WORKERS = 0` reproduce además el protocolo de
  los runs oficiales.
- `TRAIN = False`: la sección 9 **carga desde MLflow** los runs oficiales A, B y C del
  experimento `galaxy3-cnn-from-scratch` (historia, métricas y best checkpoints). Solo
  funciona donde existen esos runs. El notebook no permite entrenar en el experimento
  oficial.


In [ ]:
from galaxy3.utils import PROJECT_ROOT

# Los estudiantes no tienen los runs oficiales (viven en el MLflow local del docente):
# con TRAIN = True entrenan A, B y C en su propio experimento. TRAIN = False carga los runs
# oficiales desde MLflow y solo funciona donde existen.
TRAIN = True
EXPERIMENT_NAME = "galaxy3-estudiante"  # experimento MLflow de los runs propios
OFFICIAL_EXPERIMENT = "galaxy3-cnn-from-scratch"  # runs oficiales (cerrado, solo lectura)

# Protocolo oficial: 40 épocas. En CPU el entrenamiento es mucho más lento que en GPU;
# para practicar se puede bajar EPOCHS (por ejemplo, a 5).
EPOCHS = 40

# Workers del DataLoader (procesos que cargan y transforman imágenes en paralelo).
# NUM_WORKERS = 0 reproduce el protocolo de los runs oficiales: num_workers cambia el orden
# de los batches, así que con otro valor los resultados no son idénticos a los oficiales.
# Con workers cada época es más rápida; si hay problemas de multiprocessing, usar 0.
NUM_WORKERS = 4

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "notebook_dev"  # checkpoints, figuras y métricas

ACTIVE_EXPERIMENT = EXPERIMENT_NAME if TRAIN else OFFICIAL_EXPERIMENT

El proyecto usa su propio ambiente, `.venv`, que crea `uv sync` en la raíz del
repositorio. El intérprete del kernel debe pertenecer a él; si no, el paquete `galaxy3`
instalado no es el del proyecto.

In [ ]:
import sys
from pathlib import Path

print(sys.executable)
assert Path(sys.prefix).resolve() == (PROJECT_ROOT / ".venv").resolve(), (
    "El kernel no usa el ambiente .venv del proyecto (ver README, 'Problemas frecuentes')"
)

In [ ]:
import json
from dataclasses import replace

import mlflow
import pandas as pd
import torch
from IPython.display import display
from mlflow.tracking import MlflowClient
from torch import nn

from galaxy3.config import ExperimentConfig
from galaxy3.data import (
    build_transform,
    check_stats_match_config,
    load_dataset_stats,
    load_original_samples,
    make_split_loader,
)
from galaxy3.experiment import (
    build_model,
    find_run_id,
    load_run_result,
    load_test_evaluation,
    resolve_tracking_uri,
    run_experiment,
)
from galaxy3.inspection import capture_feature_maps
from galaxy3.metrics import collect_predictions, confusion_matrix, misclassified_indices
from galaxy3.models import count_parameters, count_trainable_parameters
from galaxy3.plots import (
    denormalize,
    plot_confusion_matrix,
    plot_feature_maps,
    plot_image_batch,
    plot_original_galaxies,
    plot_predictions,
    plot_runs_comparison,
    plot_training_curves,
)
from galaxy3.utils import get_device, set_seed

pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
config = ExperimentConfig(
    epochs=EPOCHS,
    deterministic=True,  # como los runs oficiales (docs/PILOTO.md)
    num_workers=NUM_WORKERS,  # 0 = protocolo de los runs oficiales
    mlflow_experiment_name=EXPERIMENT_NAME,
    output_root=OUTPUT_ROOT,
)
device = get_device(config.device_preference)

stats = load_dataset_stats(config.data_root)
check_stats_match_config(stats, config)  # split_seed, image_size y center_crop coinciden
mean, std = stats["mean_train"], stats["std_train"]

print("device:", device)
print("split_seed:", stats["split_seed"], "| train_seed:", config.train_seed)
print("mean_train:", [round(m, 4) for m in mean])
print("std_train: ", [round(s, 4) for s in std])
pd.DataFrame(stats["counts"]).assign(total=lambda df: df.sum(axis=1))

`mean_train` y `std_train` se calcularon **solo con train**, después de `Resize((128, 128))`
y `ToTensor()`. Usar estadísticas de validation o test filtraría información de esos
conjuntos hacia el preprocesamiento.

Hay un desbalance moderado (≈1.9:1 entre `round_smooth` y `edge_on_no_bulge`): por eso
reportamos siempre **macro F1** junto a accuracy.

## 1. Pregunta experimental y protocolo

Tres runs sobre la **misma arquitectura** y el **mismo protocolo**:

| Run | `use_batchnorm` | `dropout_p` | Lectura |
|---|---|---|---|
| A — `baseline` | `False` | 0.0 | referencia |
| B — `batchnorm` | `True` | 0.0 | A → B: cambio asociado a BatchNorm |
| C — `batchnorm_dropout` | `True` | 0.3 | B → C: cambio asociado a Dropout |

No es un diseño factorial completo (falta Dropout sin BN); se acepta por legibilidad.

**Protocolo fijo** (igual para A, B y C):

- `CrossEntropyLoss` sobre logits; `AdamW(lr=1e-3, weight_decay=0.0)`;
- **sin scheduler y sin early stopping**: mezclarlos con Dropout impediría atribuir los
  cambios observados;
- número fijo de épocas: **40**, fijado por el piloto (sección 8);
- se conserva el checkpoint de `argmin val_loss`;
- `train_seed = 2026`, `split_seed = 2026`.

**Decisiones del piloto** (`docs/PILOTO.md`), tomadas antes de ver A, B y C:

| Decisión | Valor |
|---|---|
| `center_crop` | `None` (sin `CenterCrop`) |
| `epochs` | 40 |

**Selección:** métrica primaria fijada de antemano: **macro F1 de validation en la mejor
época** (la de menor `val_loss`). El test no participa en ninguna decisión.

> No diseñamos el problema para que BatchNorm o Dropout "ganen": construimos un problema
> correcto y observamos qué ocurre. "Dropout no produjo una mejora apreciable" es una
> conclusión válida.

## 2. Datos: imágenes originales vs transformadas

Galaxy3 son tres clases de **Galaxy10 DECaLS**: `Round Smooth`, `Barred Spiral` y
`Edge-on without Bulge`, con todas sus imágenes.

Las imágenes **no son fotografías RGB**: son composiciones en **falso color** de tres
bandas fotométricas (g, r, z) asignadas a los canales de la imagen. El color ayuda a ver
la estructura, pero no es el color que vería el ojo.

Primero las originales (256×256), tal como salieron de la preparación, sin ninguna
transformación.

In [ ]:
originals = load_original_samples(config.data_root, split="train", per_class=4, seed=0)
class_names = list(originals)
first = next(iter(originals.values()))[0]
print("clases (índice = etiqueta):", class_names)
print("tamaño original:", first.size, "| modo:", first.mode)
plot_original_galaxies(originals);

Ahora **las mismas galaxias** después del pipeline de entrada al modelo:

```text
Resize((128, 128)) -> ToTensor() -> Normalize(mean_train, std_train)
```

Tras `Normalize` los valores ya no están en `[0, 1]`; para mostrarlas hay que
**desnormalizarlas** (`x * std + mean`, y `clamp(0, 1)`). Esa desnormalización es solo para
visualizar: el modelo recibe el tensor normalizado.

In [ ]:
transform = build_transform(config, mean, std)
X_same = torch.stack([transform(img) for imgs in originals.values() for img in imgs])
y_same = torch.tensor([label for label, imgs in enumerate(originals.values()) for _ in imgs])

print("tensor transformado:", tuple(X_same.shape), X_same.dtype)
print(f"rango normalizado:    [{X_same.min():.2f}, {X_same.max():.2f}]")
X_vis = denormalize(X_same, mean, std)
print(f"rango desnormalizado: [{X_vis.min():.2f}, {X_vis.max():.2f}]")
plot_image_batch(X_same, y_same, class_names, mean, std, max_images=12, ncols=4);

## 3. Batches y shapes

Un `DataLoader` agrupa imágenes en batches de forma `B × C × H × W`
(`batch × canales × alto × ancho`) y etiquetas de forma `(B,)` con enteros de clase.

Cada split tiene su propio loader. Aquí construimos **solo train y validation**
(`make_split_loader`). El loader de test solo aparece en la sección 15, para inspeccionar
los resultados de test ya registrados.

- train baraja con un `torch.Generator` sembrado con `train_seed`: misma semilla → mismo
  orden de batches;
- validation no baraja: la posición `i` de las predicciones corresponde a
  `val_dataset[i]`.

In [ ]:
train_loader = make_split_loader(config, "train", device)
val_loader = make_split_loader(config, "val", device)
val_dataset = val_loader.dataset
assert val_dataset.classes == class_names

print("train:", len(train_loader.dataset), "imágenes,", len(train_loader), "batches")
print("val:  ", len(val_dataset), "imágenes,", len(val_loader), "batches")

X, y = next(iter(train_loader))
print("X:", tuple(X.shape), X.dtype, "| y:", tuple(y.shape), y.dtype)
print("primeras etiquetas:", y[:12].tolist())
print("media por canal del batch:", X.mean(dim=(0, 2, 3)).tolist())
plot_image_batch(X, y, class_names, mean, std);

La media por canal del batch normalizado está cerca de 0: `Normalize` centra cada canal con
las estadísticas de train.

## 4. Arquitectura y parámetros

`GalaxyCNN` repite cuatro veces el mismo bloque:

```text
Conv2d(3x3, padding=1) -> BatchNorm2d | Identity -> ReLU -> MaxPool2d(2, 2)
```

- la convolución con `padding=1` conserva el alto y el ancho, y cambia el número de canales
  (3 → 32 → 64 → 128 → 256);
- `MaxPool2d(2, 2)` divide el alto y el ancho por 2;
- `AdaptiveAvgPool2d((2, 2))` lleva cualquier tamaño espacial a 2×2;
- el clasificador aplana (`256·2·2 = 1024`) y aplica `Linear(1024, 128) → ReLU →
  Dropout | Identity → Linear(128, 3)`.

El modelo devuelve **logits** (sin softmax): `CrossEntropyLoss` aplica `log_softmax`
internamente.

Las tres configuraciones son la **misma clase** con otros argumentos. `norm` es
`BatchNorm2d` o `Identity` y `dropout` es `Dropout` o `Identity`, así que los nombres de
las capas no cambian entre A, B y C.

In [ ]:
RUNS = {
    "baseline": replace(config, run_name="baseline"),
    "batchnorm": replace(config, run_name="batchnorm", use_batchnorm=True),
    "batchnorm_dropout": replace(
        config, run_name="batchnorm_dropout", use_batchnorm=True, dropout_p=0.3
    ),
}
models = {name: build_model(cfg) for name, cfg in RUNS.items()}
print(models["batchnorm_dropout"])
# Jugar aquí con los keys de RUNS para verificar arquitecturas

Recorremos las capas con **forward hooks** (`capture_feature_maps`): un hook temporal
registra la salida de una capa interna sin modificar el `forward` del modelo. Así obtenemos
la forma del tensor después de cada bloque.

In [ ]:
cpu = torch.device("cpu")
model_a = models["baseline"]
X_small = X[:4]
layers = [
    "features.block1.act",
    "features.block1.pool",
    "features.block2.pool",
    "features.block3.pool",
    "features.block4.act",
    "features.block4.pool",
    "pool",
]
shapes = {"entrada": tuple(X_small.shape)}
shapes |= {name: tuple(capture_feature_maps(model_a, X_small, name, cpu).shape) for name in layers}
model_a.eval()
with torch.inference_mode():
    shapes["logits"] = tuple(model_a(X_small).shape)
pd.Series(shapes, name="shape").to_frame()

In [ ]:
def parameter_table(models: dict[str, nn.Module]) -> pd.DataFrame:
    rows = {}
    for name, model in models.items():
        total = count_parameters(model)
        classifier = sum(p.numel() for p in model.classifier.parameters())
        rows[name] = {
            "totales": total,
            "entrenables": count_trainable_parameters(model),
            "clasificador": classifier,
            "% clasificador": 100 * classifier / total,
            "buffers (BN)": sum(b.numel() for b in model.buffers()),
        }
    counts = ["totales", "entrenables", "clasificador", "buffers (BN)"]
    return pd.DataFrame(rows).T.astype(dict.fromkeys(counts, int))


parameter_table(models)

- A tiene **520 003** parámetros y B y C **520 483**: con BatchNorm las convoluciones no
  llevan bias (la β de BN cumple ese papel) y se agregan γ y β por canal
  (`2·(32+64+128+256) = 960`, menos `480` bias quitados).
- Dropout **no agrega parámetros**: B y C tienen exactamente el mismo número.
- El clasificador concentra ≈25 % de los parámetros; el resto está en las convoluciones.
- Los buffers de BN (`running_mean`, `running_var`, `num_batches_tracked`) no son
  parámetros: no reciben gradiente.

## 5. BatchNorm: `train()` vs `eval()`

`BatchNorm2d` normaliza cada canal y luego aplica una escala γ y un desplazamiento β
aprendidos. **Qué media y varianza usa depende del modo**:

- `model.train()`: usa la media y la varianza **del batch actual**, y actualiza las
  *running statistics* (promedio móvil) en los buffers;
- `model.eval()`: usa las **running statistics** acumuladas; la salida de una imagen ya no
  depende de con qué otras imágenes viene en el batch.

Usamos un modelo B recién construido (sin entrenar) para ver el mecanismo.

In [ ]:
model_b = build_model(RUNS["batchnorm"])
bn = model_b.features.block1.norm
print("parámetros de BN:", [name for name, _ in bn.named_parameters()])
print("buffers de BN:   ", [name for name, _ in bn.named_buffers()])
print("running_mean[:4] antes:", bn.running_mean[:4].tolist())
print("running_var[:4]  antes:", bn.running_var[:4].tolist())

model_b.train()
with torch.no_grad():
    model_b(X)  # un forward en modo train actualiza las running statistics
print("running_mean[:4] después:", [round(v, 4) for v in bn.running_mean[:4].tolist()])
print("running_var[:4]  después:", [round(v, 4) for v in bn.running_var[:4].tolist()])
print("num_batches_tracked:", int(bn.num_batches_tracked))

Ahora la misma imagen (la primera del batch) dentro de dos batches distintos: uno de 16
imágenes y otro de 4.

In [ ]:
def first_logits(model: nn.Module, batch: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
        return model(batch)[0]


model_b.train()
train_16, train_4 = first_logits(model_b, X[:16]), first_logits(model_b, X[:4])
model_b.eval()
eval_16, eval_4 = first_logits(model_b, X[:16]), first_logits(model_b, X[:4])

pd.DataFrame(
    {
        "batch de 16": [train_16.tolist(), eval_16.tolist()],
        "batch de 4": [train_4.tolist(), eval_4.tolist()],
        "iguales": [torch.allclose(train_16, train_4), torch.allclose(eval_16, eval_4, atol=1e-6)],
    },
    index=["train()", "eval()"],
)

En `train()` los logits de la misma imagen cambian según el batch (la normalización usa
estadísticas del batch). En `eval()` son iguales. Por eso la evaluación siempre se hace en
modo eval, y por eso el *generalization gap* se calcula evaluando train **también en modo
eval** con el best checkpoint (no con el `train_acc` acumulado durante el entrenamiento).

Ojo: en los forwards en modo `train()` de esta celda también se actualizaron las running
statistics. Es un modelo de demostración; no se usa para nada más.

## 6. Dropout: `train()` vs `eval()`

`Dropout(p)` pone en cero cada activación con probabilidad `p` durante el entrenamiento y
escala las que sobreviven por `1/(1-p)`, para que el valor esperado no cambie. En
`eval()` no hace nada.

Primero la capa sola, sobre un tensor de unos:

In [ ]:
model_c = build_model(RUNS["batchnorm_dropout"])
dropout = model_c.classifier.dropout
print(dropout)

set_seed(0)
ones = torch.ones(1, 12)
dropout.train()
print("train():", dropout(ones))
dropout.eval()
print("eval(): ", dropout(ones))
print("1 / (1 - p) =", round(1 / (1 - dropout.p), 4))

Ahora el modelo C completo: dos forwards seguidos del mismo batch.

In [ ]:
with torch.no_grad():
    model_c.train()
    train_1, train_2 = model_c(X[:4]), model_c(X[:4])
    model_c.eval()
    eval_1, eval_2 = model_c(X[:4]), model_c(X[:4])

print("train(): dos forwards iguales?", torch.allclose(train_1, train_2))
print("eval():  dos forwards iguales?", torch.equal(eval_1, eval_2))

En `train()` cada forward usa una máscara distinta (la máscara consume el RNG global; por
eso `train_seed` también controla Dropout). En `eval()` la salida es determinista.

En C, BN está en modo train en ambos forwards con el mismo batch, así que la diferencia
entre `train_1` y `train_2` viene solo de Dropout.

## 7. Feature maps antes de entrenar

Un **feature map** es la salida de una capa convolucional para un canal: un mapa 2D que
indica dónde y con qué intensidad responde ese filtro. Los capturamos con un forward hook
temporal (`capture_feature_maps`), que se retira siempre, incluso si hay un error.

Usamos el modelo A **sin entrenar**, construido con la misma semilla que usa
`run_experiment` justo antes de construir el modelo; por eso sus pesos son los pesos
iniciales del run A. Elegimos una galaxia fija de validation y dos profundidades:

- `features.block1.act`: salida de la ReLU del bloque 1, `(32, 128, 128)`;
- `features.block3.act`: salida de la ReLU del bloque 3, `(128, 32, 32)`.

Visualización: escala de grises con **normalización por mapa** (el brillo solo se compara
dentro de un mismo mapa). Un mapa constante (por ejemplo, una ReLU apagada en toda la
imagen) se marca como "constante".

No asignamos significado a canales individuales: los feature maps sirven para ver cómo
cambia la representación con la profundidad y con el entrenamiento, no para afirmar que un
canal "detecta" una estructura astronómica.

In [ ]:
SAMPLE_INDEX = 0  # posición en val_dataset (fija para comparar antes y después)
FM_LAYERS = ["features.block1.act", "features.block3.act"]

x_sample, y_sample = val_dataset[SAMPLE_INDEX]
x_sample = x_sample.unsqueeze(0)
plot_image_batch(x_sample, torch.tensor([y_sample]), class_names, mean, std, ncols=1);

In [ ]:
set_seed(config.train_seed)
untrained_a = build_model(RUNS["baseline"])

maps_before = {
    layer: capture_feature_maps(untrained_a, x_sample, layer, cpu) for layer in FM_LAYERS
}
for layer, maps in maps_before.items():
    plot_feature_maps(maps, layer_name=f"{layer} (sin entrenar)")

Con pesos aleatorios los mapas ya "responden" a la imagen: una convolución con pesos al
azar sigue siendo un filtro local, que realza bordes y zonas brillantes. Pero no hay nada
aprendido. En el bloque 3 la resolución es menor (32×32) y cada posición resume una región
más grande de la imagen.

## 8. Qué mostró el piloto y por qué 40 épocas

Antes de los runs definitivos se entrenó **solo el baseline (A) durante 50 épocas**, en un
experimento MLflow separado (`galaxy3-pilot`), usando únicamente train y validation. El
objetivo era ver dónde converge `val_loss` y fijar un presupuesto de épocas común para A, B
y C. El piloto **no** se usó para elegir entre A, B y C.

El run del piloto solo existe en la máquina del docente. Si no está, las celdas que lo leen
se omiten con un mensaje; sus resultados están en `docs/PILOTO.md`.

In [ ]:
PILOT_JSON = PROJECT_ROOT / "outputs" / "metrics" / "pilot_baseline.json"
pilot = json.loads(PILOT_JSON.read_text()) if PILOT_JSON.is_file() else None
PILOT_SKIP = "Se omite: no están los datos del piloto (ver docs/PILOTO.md)."
if pilot is None:
    print(f"No existe {PILOT_JSON.relative_to(PROJECT_ROOT)}. {PILOT_SKIP}")
else:
    pilot_history = pilot["history"]
    plot_training_curves(pilot_history)

In [ ]:
if pilot is None:
    print(PILOT_SKIP)
else:
    pilot_epochs = [pilot_history["best_epoch"], 20, 25, 30, 40, 50]
    pilot_table = pd.DataFrame(
        {
            key: [pilot_history[key][e - 1] for e in pilot_epochs]
            for key in ("val_loss", "val_acc", "train_loss", "train_acc")
        },
        index=pd.Index(pilot_epochs, name="época"),
    ).T
    display(pilot_table)

- El mínimo de `val_loss` está en la **época 14**. Desde la época ≈22, `val_loss` sube de
  forma casi monótona hasta más del doble de su mínimo, y desde la 27 `train_acc = 1.0`: el
  modelo memoriza train.
- **Decisión: 40 épocas** para A, B y C. 40 ≈ 2.9 × 14 deja ver el sobreajuste completo en
  A, y deja margen si BatchNorm o Dropout desplazan el mínimo hasta las épocas 20–25. Más
  allá de la 30, A solo prolonga la misma tendencia; cada run cuesta ≈ 2 minutos en la
  RTX 3060.

Hay algo raro en la tabla: **`val_loss` sube pero `val_acc` no baja**. Volveremos sobre esto
en la sección 11, con nuestro propio run A.

## 9. Los tres runs

Cada run es una llamada a `run_experiment(config)`, la misma función que usa la CLI:

```bash
uv run python -m galaxy3.train --run-name batchnorm --use-batchnorm --epochs 40
```

Dentro de `run_experiment`: siembra `train_seed` → construye los loaders de train y val →
construye el modelo → entrena `epochs` épocas con `fit` (guardando el checkpoint de menor
`val_loss`) → recarga ese checkpoint → evalúa train y val **en modo eval** → calcula métricas
y figuras → registra todo en MLflow. **No toca test.**

Los runs se ejecutan **en serie**, para que los tiempos por época sean comparables. (Como
procesos independientes también podrían lanzarse en paralelo desde varias terminales.)

**Con `TRAIN = False`** no se entrena. Para cada run, `find_run_id` busca en
`galaxy3-cnn-from-scratch` el único run `FINISHED` con ese nombre, y `load_run_result`
reconstruye su resultado:

- la historia sale de las métricas por época registradas en MLflow;
- las evaluaciones de train y val, del JSON de métricas registrado como artifact;
- el best checkpoint se descarga de los artifacts del run.

Se comprueba que esas fuentes coincidan entre sí. El resultado es el mismo tipo de objeto
que devuelve `run_experiment`, así que el resto del notebook funciona igual en los dos
modos.


In [ ]:
print(f"TRAIN = {TRAIN} | experimento MLflow: {ACTIVE_EXPERIMENT}")
results = {}
if TRAIN:
    assert EXPERIMENT_NAME != OFFICIAL_EXPERIMENT, "El experimento oficial está cerrado"
    print(f"EPOCHS = {EPOCHS} | NUM_WORKERS = {NUM_WORKERS} | salida: {OUTPUT_ROOT}")
    for name, run_config in RUNS.items():
        print(f"\n=== {name} ===")
        results[name] = run_experiment(run_config)
else:
    for name in RUNS:
        results[name] = load_run_result(find_run_id(OFFICIAL_EXPERIMENT, name))
        cfg = results[name].config
        print(
            f"{name}: run_id {results[name].run_id} | epochs={cfg.epochs}, "
            f"deterministic={cfg.deterministic}, num_workers={cfg.num_workers}"
        )

In [ ]:
summary = pd.DataFrame({name: result.final_metrics() for name, result in results.items()}).T
summary["best_epoch"] = summary["best_epoch"].astype(int)
summary["s/época"] = [
    sum(r.history["epoch_time"]) / len(r.history["epoch_time"]) for r in results.values()
]
summary

## 10. Curvas y comparación en validation

Cómo leer las curvas de cada run:

- **`train_*`** se acumula durante la época en modo train (con Dropout activo, BN con
  estadísticas del batch y pesos que cambian dentro de la época); **`val_*`** se mide al
  final de la época en modo eval. Por eso no son directamente comparables punto a punto.
- La línea discontinua marca la **mejor época** (mínimo de `val_loss`): es el checkpoint que
  se conserva.
- Si `val_loss` sube después del mínimo mientras `train_loss` sigue bajando, el modelo
  sobreajusta.

In [ ]:
for name, result in results.items():
    fig = plot_training_curves(result.history)
    fig.suptitle(f"{name} — {fig.get_suptitle()}")

Las tres historias de validation superpuestas. Cada run tiene un color y un estilo de
línea fijos (continua, discontinua, punteada), así que se distinguen también en escala de
grises. El marcador indica la mejor época de cada run; en el panel de accuracy marca la
`val_acc` del checkpoint conservado, que no tiene por qué ser el máximo de `val_acc`.

In [ ]:
plot_runs_comparison({name: result.history for name, result in results.items()});

In [ ]:
for name, result in results.items():
    plot_confusion_matrix(
        result.val_eval.report.confusion_matrix,
        class_names,
        title=f"{name} — validation, mejor época ({result.history['best_epoch']})",
    )

In [ ]:
def comparison_row(result) -> dict[str, float]:
    h = result.history
    best = h["best_epoch"]
    return {
        "best_epoch": best,
        "val_loss (mejor)": h["best_val_loss"],
        "val_loss (última)": h["val_loss"][-1],
        "val_acc (mejor)": result.val_eval.report.accuracy,
        "val_acc (máx.)": max(h["val_acc"]),
        "val_macro_f1": result.val_eval.report.macro_f1,
        "train_acc_eval": result.train_eval.report.accuracy,
        "gap": result.generalization_gap,
    }


comparison = pd.DataFrame({name: comparison_row(r) for name, r in results.items()}).T
comparison["best_epoch"] = comparison["best_epoch"].astype(int)
comparison

In [ ]:
selected = comparison["val_macro_f1"].idxmax()
n_val = results[selected].val_samples
print(f"Configuración seleccionada (macro F1 de validation): {selected}")
print(f"Con {n_val} imágenes de validation, 1 punto de accuracy ≈ {n_val / 100:.0f} imágenes.")

**Cómo interpretar la tabla**

- `gap = train_acc_eval − val_acc`, ambos en modo eval con el best checkpoint.
- Comparamos con **una sola `train_seed`**: las diferencias son descriptivas, no evidencia
  estadística. Una diferencia de un punto son ≈9 imágenes de validation.
- La selección usa la métrica primaria fijada antes de correr (macro F1 de validation en la
  mejor época). El test no participa.
- Fíjense en el run A: `val_loss (última)` frente a `val_loss (mejor)`, y `val_acc (máx.)`
  frente a `val_acc (mejor)`. Es la misma paradoja del piloto.

## 11. Loss vs accuracy: sobreconfianza y errores

**La paradoja.** En el run A (y en el piloto), después de la mejor época `val_loss` sube con
fuerza, pero `val_acc` se mantiene o incluso mejora. ¿Cómo puede empeorar la loss si el
modelo no se equivoca más?

**La clave: qué mide cada una.** Para una imagen con clase real `c`, la entropía cruzada es

$$\ell = -\log p_c, \qquad p = \mathrm{softmax}(\text{logits}).$$

- La **accuracy** solo mira si `argmax p` es la clase real: no distingue un acierto con
  `p_c = 0.51` de uno con `p_c = 0.99`, ni un error dudoso de uno seguro.
- La **loss** mira la probabilidad asignada a la clase real. Un acierto confiado cuesta casi
  0, pero un **error confiado** cuesta mucho: con `p_c = 0.01`, `ℓ = 4.6`; con
  `p_c = 0.0001`, `ℓ = 9.2`. La loss no tiene techo.

Ejemplo de juguete: dos modelos con **las mismas predicciones** (mismo `argmax`, misma
accuracy) sobre tres imágenes de clase real 0.

In [ ]:
y_toy = torch.tensor([0, 0, 0])
toy_logits = {
    "prudente": torch.tensor([[2.0, 0.0, 0.0], [1.5, 0.5, 0.0], [0.0, 0.8, 0.0]]),
    "sobreconfiado": torch.tensor([[9.0, 0.0, 0.0], [8.0, 0.0, 0.0], [0.0, 9.0, 0.0]]),
}
per_sample_loss = nn.CrossEntropyLoss(reduction="none")

rows = {}
for name, logits in toy_logits.items():
    losses = per_sample_loss(logits, y_toy)
    rows[name] = {
        "pred": logits.argmax(dim=1).tolist(),
        "p(clase real)": [round(p, 4) for p in torch.softmax(logits, dim=1)[:, 0].tolist()],
        "loss por imagen": [round(v, 3) for v in losses.tolist()],
        "accuracy": (logits.argmax(dim=1) == y_toy).float().mean().item(),
        "loss media": losses.mean().item(),
    }
pd.DataFrame(rows).T

Ambos aciertan 2 de 3. El sobreconfiado tiene una loss media mucho mayor, y casi toda viene
de su **único error**, cometido con mucha seguridad.

Eso es lo que pasa después de la mejor época: el modelo sigue empujando los logits de train
hacia `train_loss → 0` y se vuelve más seguro **en todo**, también en los errores que todavía
comete en validation. `val_acc` no cambia (los errores son casi los mismos), pero cada error
pesa más en `val_loss`.

**Consecuencia para el protocolo.** Elegimos el checkpoint por `argmin val_loss` (fijado de
antemano): es el modelo que asigna **mejores probabilidades** a la clase real (menor
entropía cruzada en validation), no el de mayor accuracy. En el piloto eso
significó ≈2.7 puntos menos de `val_acc` que en épocas tardías. No cambiamos el criterio a
posteriori; lo discutimos.

### Lo mismo en nuestros runs

Recargamos el best checkpoint de cada run **explícitamente** con
`torch.load(..., weights_only=True)`: el archivo solo contiene tensores, `state_dict` y tipos
primitivos, así que no hace falta (ni conviene) permitir objetos arbitrarios de Python.

In [ ]:
def load_best_model(result) -> tuple[nn.Module, dict]:
    checkpoint = torch.load(result.checkpoint_path, map_location=device, weights_only=True)
    model = build_model(result.config).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    return model, checkpoint


best_models = {}
for name, result in results.items():
    best_models[name], checkpoint = load_best_model(result)
    print(
        f"{name}: época {checkpoint['epoch']}, val_loss {checkpoint['val_loss']:.4f}, "
        f"claves {sorted(checkpoint)}"
    )

Con `collect_predictions` (modo eval, mismo bucle que `engine.evaluate`) obtenemos las
probabilidades de validation y separamos `−log p(clase real)` entre aciertos y errores.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
val_predictions = {
    name: collect_predictions(model, val_loader, loss_fn, device)
    for name, model in best_models.items()
}


def loss_breakdown(pred) -> dict[str, float]:
    p_true = pred.probabilities[torch.arange(len(pred.y_true)), pred.y_true]
    nll = -torch.log(p_true.clamp_min(torch.finfo(p_true.dtype).tiny))
    wrong = pred.y_pred != pred.y_true
    return {
        "val_loss": pred.loss,
        "val_acc": pred.accuracy,
        "errores": int(wrong.sum()),
        "−log p media (aciertos)": nll[~wrong].mean().item(),
        "−log p media (errores)": nll[wrong].mean().item() if wrong.any() else float("nan"),
        "% de la loss por errores": 100 * nll[wrong].sum().item() / nll.sum().item(),
        "confianza media en errores": pred.probabilities[wrong].max(dim=1).values.mean().item()
        if wrong.any()
        else float("nan"),
    }


breakdown = pd.DataFrame({name: loss_breakdown(p) for name, p in val_predictions.items()}).T
breakdown.astype({"errores": int})

Los errores son pocos, pero aportan una fracción grande de la loss: cada uno cuesta mucho
más que un acierto. Este desglose es en la **mejor época**; después de ella, la confianza en
esos errores sigue creciendo y `val_loss` sube aunque `val_acc` no cambie.

### Errores de la configuración seleccionada

Las imágenes de validation peor clasificadas por el run seleccionado, ordenadas de mayor a
menor confianza en el error. Las probabilidades salen de un softmax aplicado solo para el
análisis; el modelo sigue devolviendo logits.

In [ ]:
pred = val_predictions[selected]
error_idx = misclassified_indices(pred, max_count=12)
print(
    f"{selected}: {int((pred.y_pred != pred.y_true).sum())} errores en {len(pred.y_true)} imágenes"
)
if len(error_idx) > 0:
    X_errors = torch.stack([val_dataset[int(i)][0] for i in error_idx])
    plot_predictions(
        X_errors,
        pred.y_true[error_idx],
        pred.probabilities[error_idx],
        class_names,
        mean,
        std,
        title=f"{selected}: errores en validation (más confiados primero)",
    )

Preguntas para discutir: ¿entre qué clases se concentran los errores (ver las matrices de
confusión de la sección 10)? ¿Hay imágenes ambiguas incluso para un humano: galaxias pequeñas,
con poca señal o con estrellas cercanas? ¿Con qué confianza se equivoca el modelo?

## 12. MLflow

MLflow guarda, para cada run: los **parámetros** (configuración, device, número de
parámetros, tamaños de los splits), las **métricas por época** (`step = epoch`), las
**métricas finales** y los **artifacts** (config, best checkpoint, curvas, tabla de métricas).
Las figuras las construye `plots.py` con Matplotlib; `experiment.py` las registra.

El backend local es la base SQLite `mlflow.db` en la raíz del repositorio (o
`MLFLOW_TRACKING_URI` si está definida).

In [ ]:
tracking_uri = resolve_tracking_uri()
mlflow.set_tracking_uri(tracking_uri)
print("tracking URI:", tracking_uri)

run_ids = [result.run_id for result in results.values()]
runs = mlflow.search_runs(experiment_names=[ACTIVE_EXPERIMENT])
columns = [
    "tags.mlflow.runName",
    "run_id",
    "status",
    "params.use_batchnorm",
    "params.dropout_p",
    "params.epochs",
    "metrics.best_epoch",
    "metrics.val_acc_best",
    "metrics.val_macro_f1",
    "metrics.generalization_gap",
]
runs[runs["run_id"].isin(run_ids)][columns]

In [ ]:
client = MlflowClient()
example_run = results[selected].run_id
print("artifacts del run", selected)
for artifact in client.list_artifacts(example_run):
    print("  ", artifact.path, "(carpeta)" if artifact.is_dir else "")

La historia por época también se puede recuperar desde MLflow. Por ejemplo, `val_loss` del
piloto (experimento `galaxy3-pilot`), que debe coincidir con la del JSON de la sección 8:

In [ ]:
if pilot is None:
    print(PILOT_SKIP)
elif not mlflow.search_runs(
    experiment_names=["galaxy3-pilot"],
    filter_string=f"attributes.run_id = '{pilot['run_id']}'",
    output_format="list",
):
    print(f"El run del piloto no está en este MLflow. {PILOT_SKIP}")
else:
    pilot_val_loss = (
        pd.DataFrame(
            [(m.step, m.value) for m in client.get_metric_history(pilot["run_id"], "val_loss")],
            columns=["época", "val_loss"],
        )
        .sort_values("época")
        .set_index("época")
    )
    assert len(pilot_val_loss) == len(pilot_history["val_loss"])
    display(pilot_val_loss.T)

Para comparar los runs gráficamente, abrir la interfaz de MLflow desde una terminal en la
raíz del repositorio (y luego <http://127.0.0.1:5000>):

In [ ]:
print(f"uv run mlflow ui --backend-store-uri {tracking_uri}")

## 13. Feature maps después de entrenar

La misma galaxia, las mismas capas y el **mismo modelo A**, ahora con los pesos del best
checkpoint. Como el modelo sin entrenar de la sección 7 tenía los pesos iniciales del run A,
la única diferencia es el entrenamiento.

In [ ]:
trained_a = best_models["baseline"]
maps_after = {
    layer: capture_feature_maps(trained_a, x_sample, layer, device) for layer in FM_LAYERS
}

for layer in FM_LAYERS:
    plot_feature_maps(maps_before[layer], layer_name=f"{layer} — antes de entrenar")
    plot_feature_maps(maps_after[layer], layer_name=f"{layer} — después de entrenar (A)")

In [ ]:
def activation_summary(maps: torch.Tensor) -> dict[str, float]:
    per_channel_max = maps[0].amax(dim=(1, 2))
    return {
        "canales": maps.shape[1],
        "canales apagados (ReLU = 0 en todo el mapa)": int((per_channel_max == 0).sum()),
        "fracción de activaciones = 0": (maps == 0).float().mean().item(),
        "activación media": maps.mean().item(),
    }


pd.DataFrame(
    {
        f"{layer} — {moment}": activation_summary(maps[layer])
        for layer in FM_LAYERS
        for moment, maps in (("antes", maps_before), ("después", maps_after))
    }
).T.astype({"canales": int, "canales apagados (ReLU = 0 en todo el mapa)": int})

Qué mirar (sin asignar significado a canales concretos):

- ¿Cambió la **estructura** de los mapas? Con pesos aleatorios los mapas del bloque 1 se
  parecen a versiones filtradas de la imagen; después de entrenar, ¿los canales responden
  de forma más diversa (centro, bordes, fondo)?
- ¿Cambió la **dispersión** (fracción de ceros, canales apagados)? En el bloque 3 la
  representación es más abstracta y de menor resolución: cada posición resume una región
  amplia de la galaxia.
- Las figuras usan normalización visual **por mapa**: el brillo no se compara entre mapas
  ni entre antes y después; para eso está la tabla.

## 14. Cierre

Preguntas para discutir con los runs oficiales (resultados en `docs/RESULTADOS.md`):

1. **A → B (BatchNorm):** ¿cambió la mejor época, la forma de las curvas o el macro F1 de
   validation? ¿Cambió el generalization gap?
2. **B → C (Dropout):** ¿cambió el sobreajuste después del mínimo de `val_loss`? ¿Hubo una
   mejora apreciable en validation, o la diferencia está dentro de unas pocas imágenes?
3. **Loss vs accuracy:** ¿aparece en B y C la misma sobreconfianza que en A?

**Límites de esta comparación:**

- una sola `train_seed`, así que las diferencias son descriptivas;
- no es un diseño factorial completo (falta Dropout sin BN);
- la selección por `val_loss` favorece las probabilidades bien asignadas sobre la accuracy.

**Evaluación final:** con la configuración elegida en validation (C), `evaluate_on_test` se
ejecutó **una sola vez** en la Fase 13 y registró los resultados en el mismo run de MLflow.
La sección 15 los lee de ahí.


## 15. Evaluación final en test (resultados registrados)

La configuración seleccionada con validation es **C (`batchnorm_dropout`)**. En la Fase 13,
`evaluate_on_test` se ejecutó **una sola vez** sobre su best checkpoint, y los resultados
quedaron registrados en el mismo run de MLflow (carpeta `test/`).

Aquí **no se vuelve a evaluar**: `load_test_evaluation` lee lo registrado y comprueba que
coincida con las métricas de test de MLflow. Si el run no tuviera evaluación de test,
lanzaría un error en lugar de evaluar.

Ver test ahora no cambia nada: la selección ya estaba hecha con validation, y A y B no
tienen métricas de test. Esta sección usa siempre los runs oficiales, sea cual sea el
valor de `TRAIN`. Si no existen en este MLflow (por ejemplo, en la máquina de un
estudiante), las celdas se omiten con un mensaje; los resultados de test están en
`docs/RESULTADOS.md`.


In [ ]:
FINAL_RUN = "batchnorm_dropout"  # seleccionada con validation (docs/RESULTADOS.md)
TEST_SKIP = "Se omite: no están los runs oficiales (ver docs/RESULTADOS.md)."
try:
    final_run_id = find_run_id(OFFICIAL_EXPERIMENT, FINAL_RUN)
except ValueError as err:  # no existe el experimento oficial o el run de C
    final_run_id = None
    print(f"{err}. {TEST_SKIP}")

if final_run_id is not None:
    official_c = load_run_result(final_run_id)  # con TRAIN = False es results[FINAL_RUN]
    test_result = load_test_evaluation(final_run_id)  # lee lo registrado; no evalúa

    val_report, test_report = official_c.val_eval.report, test_result.test_eval.report
    n_errors = {
        split: int(report.confusion_matrix.sum() - report.confusion_matrix.trace())
        for split, report in (("validation", val_report), ("test", test_report))
    }
    print(f"run {final_run_id} | checkpoint de la época {test_result.checkpoint_epoch}")
    print(f"errores: {n_errors} (de {test_result.test_samples} imágenes en cada split)")

    summary_test = pd.DataFrame(
        {
            "validation": [official_c.val_eval.loss, val_report.accuracy, val_report.macro_f1],
            "test": [test_result.test_eval.loss, test_report.accuracy, test_report.macro_f1],
        },
        index=["loss", "accuracy", "macro F1"],
    )
    summary_test["test − val"] = summary_test["test"] - summary_test["validation"]
    display(summary_test)

In [ ]:
if final_run_id is None:
    print(TEST_SKIP)
else:
    display(
        pd.DataFrame(
            {
                (metric, split): getattr(report, metric).tolist()
                for metric in ("precision", "recall", "f1")
                for split, report in (("val", val_report), ("test", test_report))
            },
            index=test_result.class_names,
        )
    )

In [ ]:
if final_run_id is None:
    print(TEST_SKIP)
else:
    for report, split in ((val_report, "validation"), (test_report, "test")):
        plot_confusion_matrix(
            report.confusion_matrix,
            test_result.class_names,
            title=f"C — {split} (best checkpoint, época {test_result.checkpoint_epoch})",
        )

Las métricas globales de test quedan a menos de una imagen de las de validation. Lo que
cambia es **la forma del error**: casi todos los errores de test son espirales barradas
predichas como `round_smooth`, mucho más asimétrico que en validation. El recall de
`barred_spiral` baja y el de `round_smooth` sube.

### Las espirales barradas que C confunde con `round_smooth`

MLflow guarda la matriz de confusión de test, pero no **qué** imágenes se clasificaron mal.
Para verlas, pasamos el best checkpoint de C, congelado y en modo eval, por las imágenes de
test con `collect_predictions`. No es una nueva evaluación: no se registra nada y no se
decide nada. Como control, la matriz de confusión obtenida tiene que ser **idéntica** a la
registrada.


In [ ]:
if final_run_id is None:
    print(TEST_SKIP)
else:
    c_model = build_model(official_c.config).to(device)
    c_checkpoint = torch.load(official_c.checkpoint_path, map_location=device, weights_only=True)
    c_model.load_state_dict(c_checkpoint["model_state_dict"])
    assert c_checkpoint["epoch"] == test_result.checkpoint_epoch

    test_loader = make_split_loader(official_c.config, "test", device)
    test_dataset = test_loader.dataset
    assert test_dataset.classes == test_result.class_names
    test_pred = collect_predictions(c_model, test_loader, nn.CrossEntropyLoss(), device)

    reproduced = confusion_matrix(test_pred.y_true, test_pred.y_pred, len(test_dataset.classes))
    assert torch.equal(reproduced, test_report.confusion_matrix), (
        "no reproduce la matriz registrada"
    )
    print("matriz de confusión reproducida = matriz registrada en MLflow")
    print(reproduced.tolist())

In [ ]:
if final_run_id is None:
    print(TEST_SKIP)
else:
    BARRED = test_dataset.classes.index("barred_spiral")
    ROUND = test_dataset.classes.index("round_smooth")

    barred_as_round = torch.nonzero(
        (test_pred.y_true == BARRED) & (test_pred.y_pred == ROUND)
    ).flatten()
    confidence = test_pred.probabilities[barred_as_round, ROUND]
    barred_as_round = barred_as_round[torch.argsort(confidence, descending=True, stable=True)]
    print(f"barred_spiral → round_smooth en test: {len(barred_as_round)} imágenes")
    print(
        "p(round_smooth):", [round(p, 2) for p in confidence.sort(descending=True).values.tolist()]
    )

    plot_predictions(
        torch.stack([test_dataset[int(i)][0] for i in barred_as_round]),
        test_pred.y_true[barred_as_round],
        test_pred.probabilities[barred_as_round],
        test_dataset.classes,
        mean,
        std,
        max_images=len(barred_as_round),
        ncols=6,
        title="C en test: barred_spiral predichas como round_smooth (más confiadas primero)",
    )

Preguntas para discutir, sin reabrir la selección:

- ¿Se distingue la barra a 128×128? ¿Hay galaxias pequeñas, con poca señal o vistas de
  frente, en las que la barra apenas se nota?
- ¿Con qué confianza se equivoca el modelo? ¿Hay errores muy seguros, o casi todos están
  cerca de 0.5?
- Las etiquetas de Galaxy10 DECaLS vienen de votos de voluntarios (Galaxy Zoo). ¿Hay
  imágenes en las que la etiqueta `barred_spiral` parece discutible?
- Como validation solo tenía 15 errores de este tipo, la asimetría de test **no se podía
  anticipar** desde validation. Con una sola semilla y un solo split, tampoco sabemos si es
  estable.
